# 6.2 CNN 卷积核与特征图可视化

本 notebook 介绍如何可视化 CNN 的卷积核（kernel）和特征图（feature map），帮助理解卷积神经网络的工作原理。

**学习目标：**
- 掌握 `make_grid` 函数的使用
- 学会提取和可视化卷积核
- 学会通过手动前向和 hook 两种方式提取特征图
- 理解特征图的统计特性

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

print(f"PyTorch 版本: {torch.__version__}")

## 1. make_grid 函数

`torchvision.utils.make_grid` 将一批图像排列成网格，方便可视化。

**关键参数：**
- `tensor`: (N, C, H, W) 格式的图像批次
- `nrow`: 每行显示的图像数量
- `normalize`: 是否归一化到 [0, 1]
- `padding`: 图像之间的间距（像素）

In [ ]:
# 创建一批随机图像
images = torch.rand(16, 3, 32, 32)  # 16 张 RGB 图像

# 使用 make_grid 排列
grid_default = make_grid(images, nrow=8)  # 每行 8 张
print(f"输入形状: {images.shape}")
print(f"网格形状: {grid_default.shape}  (C, H, W)")

# 不同参数对比
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 默认参数
grid1 = make_grid(images, nrow=8, padding=2)
axes[0].imshow(grid1.permute(1, 2, 0).numpy())
axes[0].set_title("nrow=8, padding=2")
axes[0].axis("off")

# 每行 4 张
grid2 = make_grid(images, nrow=4, padding=2)
axes[1].imshow(grid2.permute(1, 2, 0).numpy())
axes[1].set_title("nrow=4, padding=2")
axes[1].axis("off")

# 大间距
grid3 = make_grid(images, nrow=8, padding=5)
axes[2].imshow(grid3.permute(1, 2, 0).numpy())
axes[2].set_title("nrow=8, padding=5")
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# normalize 参数: 将值范围映射到 [0, 1]
# 当图像值不在 [0, 1] 范围时非常有用
images_negative = torch.randn(8, 3, 32, 32)  # 包含负值
print(f"值范围: [{images_negative.min():.2f}, {images_negative.max():.2f}]")

fig, axes = plt.subplots(1, 2, figsize=(12, 3))

# 不归一化（负值会被截断）
grid_raw = make_grid(images_negative, nrow=4, normalize=False)
axes[0].imshow(grid_raw.permute(1, 2, 0).numpy().clip(0, 1))
axes[0].set_title("normalize=False (负值截断)")
axes[0].axis("off")

# 归一化到 [0, 1]
grid_norm = make_grid(images_negative, nrow=4, normalize=True)
axes[1].imshow(grid_norm.permute(1, 2, 0).numpy())
axes[1].set_title("normalize=True (映射到 [0,1])")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 2. 定义示例 CNN 模型

定义一个简单的 CNN 用于后续的卷积核和特征图可视化。

In [ ]:
class DemoCNN(nn.Module):
    """用于可视化演示的 CNN 模型。"""
    
    def __init__(self):
        super().__init__()
        # 第一层: 3 通道输入 -> 16 个卷积核
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        
        # 第二层: 16 -> 32
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        
        # 第三层: 32 -> 64
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(64 * 4 * 4, 10)
    
    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # 32 -> 16
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # 16 -> 8
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  # 8 -> 4
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

model = DemoCNN()
model.eval()  # 设为评估模式

print("模型结构:")
for name, module in model.named_children():
    print(f"  {name}: {module}")

## 3. 提取和可视化卷积核

卷积核的形状为 `(out_channels, in_channels, kH, kW)`。

对于第一层卷积（输入为 RGB），每个卷积核有 3 个通道，可以直接作为 RGB 图像显示。

In [ ]:
# 获取第一层卷积核
conv1_weights = model.conv1.weight.data.clone()
print(f"conv1 卷积核形状: {conv1_weights.shape}")
print(f"  含义: (out_channels={conv1_weights.shape[0]}, "
      f"in_channels={conv1_weights.shape[1]}, "
      f"kH={conv1_weights.shape[2]}, kW={conv1_weights.shape[3]})")
print(f"  值范围: [{conv1_weights.min():.4f}, {conv1_weights.max():.4f}]")

In [ ]:
# 可视化第一层卷积核（作为 RGB 图像）
# conv1_weights 形状: (16, 3, 3, 3) -> 16 个 RGB 卷积核
kernel_grid = make_grid(conv1_weights, nrow=8, normalize=True, padding=1)

plt.figure(figsize=(12, 3))
plt.imshow(kernel_grid.permute(1, 2, 0).numpy())
plt.title("Conv1 卷积核 (16 个 3x3 RGB 卷积核)")
plt.axis("off")
plt.show()

print("说明: 每个小方块是一个 3x3 的 RGB 卷积核")
print("颜色和纹理代表该卷积核对不同颜色和方向的敏感程度")

In [ ]:
# 可视化深层卷积核（逐通道灰度显示）
conv2_weights = model.conv2.weight.data.clone()
print(f"conv2 卷积核形状: {conv2_weights.shape}")
print("深层卷积核有多个输入通道，无法直接作为 RGB 显示")
print("通常选择某些卷积核的特定通道进行灰度可视化")

# 取前 16 个卷积核的第 0 个输入通道
kernels_ch0 = conv2_weights[:16, 0:1, :, :]  # (16, 1, 3, 3)
kernel_grid = make_grid(kernels_ch0, nrow=8, normalize=True, padding=1)

plt.figure(figsize=(12, 3))
plt.imshow(kernel_grid[0].numpy(), cmap="gray")
plt.title("Conv2 卷积核 (前 16 个, 第 0 通道, 灰度)")
plt.axis("off")
plt.show()

## 4. 提取特征图 - 方法一：手动逐层前向

最直观的方式：手动执行每一层，保存中间结果。

缺点：需要了解模型内部结构，且代码耦合度高。

In [ ]:
# 创建测试输入
test_input = torch.randn(1, 3, 32, 32)

# 手动逐层前向传播
with torch.no_grad():
    # 第一层
    out1 = model.conv1(test_input)
    out1_act = F.relu(model.bn1(out1))
    out1_pool = model.pool(out1_act)
    
    # 第二层
    out2 = model.conv2(out1_pool)
    out2_act = F.relu(model.bn2(out2))
    out2_pool = model.pool(out2_act)
    
    # 第三层
    out3 = model.conv3(out2_pool)
    out3_act = F.relu(model.bn3(out3))
    out3_pool = model.pool(out3_act)

print("各层特征图形状:")
print(f"  conv1 输出: {out1_act.shape}")
print(f"  pool1 输出: {out1_pool.shape}")
print(f"  conv2 输出: {out2_act.shape}")
print(f"  pool2 输出: {out2_pool.shape}")
print(f"  conv3 输出: {out3_act.shape}")
print(f"  pool3 输出: {out3_pool.shape}")

In [ ]:
# 可视化第一层特征图
# out1_act 形状: (1, 16, 32, 32) -> 取第 0 个样本
feature_maps = out1_act[0]  # (16, 32, 32)

# 将每个通道作为单独的灰度图
feature_maps_display = feature_maps.unsqueeze(1)  # (16, 1, 32, 32)
grid = make_grid(feature_maps_display, nrow=8, normalize=True, padding=1)

plt.figure(figsize=(14, 4))
plt.imshow(grid[0].numpy(), cmap="viridis")
plt.title("Conv1 特征图 (16 通道, 手动提取)")
plt.axis("off")
plt.colorbar(shrink=0.6)
plt.show()

## 5. 提取特征图 - 方法二：使用 forward hook（推荐）

使用 `register_forward_hook` 注册钩子函数，在前向传播时自动捕获中间层输出。

**优点：**
- 不需要修改模型代码
- 不需要了解模型内部结构的细节
- 可以同时捕获多层的输出

In [ ]:
# 使用 hook 提取特征图
class FeatureExtractor:
    """使用 forward hook 提取指定层的特征图。"""
    
    def __init__(self, model: nn.Module, target_layers: list):
        self.features = {}
        self.hooks = []
        
        for name in target_layers:
            # 获取目标层
            layer = dict(model.named_modules())[name]
            # 注册 hook
            hook = layer.register_forward_hook(self._make_hook(name))
            self.hooks.append(hook)
    
    def _make_hook(self, name: str):
        """创建 hook 函数，捕获输出。"""
        def hook_fn(module, input, output):
            self.features[name] = output.detach()
        return hook_fn
    
    def remove_hooks(self):
        """移除所有 hook。"""
        for hook in self.hooks:
            hook.remove()
        self.hooks.clear()

# 指定要提取的层
target_layers = ["conv1", "conv2", "conv3"]
extractor = FeatureExtractor(model, target_layers)

# 正常前向传播，hook 会自动捕获
with torch.no_grad():
    output = model(test_input)

# 查看提取到的特征图
for name, feat in extractor.features.items():
    print(f"层 {name}: 特征图形状 {feat.shape}")

# 移除 hook（使用完毕后要移除）
extractor.remove_hooks()

In [ ]:
# 可视化所有层的特征图
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, feat) in enumerate(extractor.features.items()):
    # 取第 0 个样本，每个通道作为灰度图
    fmap = feat[0]  # (C, H, W)
    n_channels = min(fmap.shape[0], 16)  # 最多显示 16 个通道
    fmap_display = fmap[:n_channels].unsqueeze(1)  # (n, 1, H, W)
    grid = make_grid(fmap_display, nrow=4, normalize=True, padding=1)
    
    axes[idx].imshow(grid[0].numpy(), cmap="viridis")
    axes[idx].set_title(f"{name}: {feat.shape[1]}ch x {feat.shape[2]}x{feat.shape[3]}")
    axes[idx].axis("off")

plt.suptitle("各层特征图对比 (hook 方式提取)", fontsize=14)
plt.tight_layout()
plt.show()

print("观察: 随着层数加深，特征图尺寸缩小，通道数增加")
print("浅层特征: 边缘、纹理等低级特征")
print("深层特征: 语义更抽象的高级特征")

## 6. 特征图统计信息

分析特征图的统计特性，帮助诊断模型训练状态。

In [ ]:
# 重新提取特征图
extractor = FeatureExtractor(model, target_layers)
with torch.no_grad():
    _ = model(test_input)

print("特征图统计信息:")
print(f"{'层名':<8} {'形状':<20} {'均值':>8} {'标准差':>8} {'最小值':>8} {'最大值':>8} {'零值比例':>10}")
print("-" * 80)

for name, feat in extractor.features.items():
    fmap = feat[0]  # 取第 0 个样本
    shape_str = str(tuple(fmap.shape))
    mean_val = fmap.mean().item()
    std_val = fmap.std().item()
    min_val = fmap.min().item()
    max_val = fmap.max().item()
    zero_ratio = (fmap == 0).float().mean().item()
    
    print(f"{name:<8} {shape_str:<20} {mean_val:>8.4f} {std_val:>8.4f} "
          f"{min_val:>8.4f} {max_val:>8.4f} {zero_ratio:>10.2%}")

extractor.remove_hooks()

print("\n分析要点:")
print("- 均值接近 0: BatchNorm 生效")
print("- 零值比例: ReLU 激活后的死神经元比例")
print("- 标准差过大/过小: 可能存在梯度爆炸/消失")

## 7. 总结

### make_grid 函数
- 将批量图像排列成网格，方便一次性显示多张图像
- `normalize=True` 将值映射到 [0, 1]，适用于含负值的特征图
- `nrow` 控制每行图像数，`padding` 控制间距

### 卷积核可视化
- 通过 `model.conv1.weight.data` 获取卷积核参数
- 第一层卷积核可直接作为 RGB 图像显示（3 个输入通道）
- 深层卷积核需要选择特定通道进行灰度显示

### 特征图提取

| 方法 | 优点 | 缺点 |
|------|------|------|
| 手动逐层前向 | 直观，完全控制 | 需要了解模型细节，代码耦合 |
| forward hook | 非侵入式，灵活 | 需要理解 hook 机制 |

### 关键观察
1. 浅层特征图保留空间分辨率，捕捉边缘、纹理等低级特征
2. 深层特征图分辨率降低但通道数增加，捕捉语义更丰富的特征
3. 特征图统计信息可用于诊断模型训练状态

---

## 练习题

**练习1（基础）：** 定义一个 4 层 CNN（通道数为 3->8->16->32->64），可视化每一层的卷积核。

**练习2（进阶）：** 使用 `FeatureExtractor` 类提取上述 CNN 所有卷积层的特征图，并计算每层的统计信息。分析从浅层到深层的特征变化趋势。

**练习3（挑战）：** 对比两张不同的输入图像（一张全零，一张随机）通过同一个 CNN 后的特征图差异。思考：全零输入的特征图为什么不全为零？